In [ ]:
import pandas as pd
# Load input CSV
df = pd.read_csv("ADDRESS")
df

In [ ]:
import os
import time
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import csv

# ✅ Use the best model: LLaMA 3 - 70B Instruct --- can't use it ..gets OOM
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
from huggingface_hub import login
login("YOUR_KEY")

# ✅ Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
import csv
from tqdm import tqdm
# Define output CSV and write header
output_file = "ADDRESS"
with open(output_file, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([
        'qid', 'context','label' ,'original_question', 'suggested_question','llama_answer_uq','llama_answer_sq'
    ])


def generate_vote(prompt, max_new_tokens=24):
    # Tokenize and move to model device
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Run greedy decoding (no sampling)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,              
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode full output
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Clean: remove prompt prefix and keep first valid line
    rewritten = decoded[len(prompt):].strip().split("\n")

 

    return rewritten

# Iterate over rows
for i, row in tqdm(df.iterrows(), total=len(df)):
    qid = row['qid']
    context = row['context']
    label = row['label']
    uq = row['UQ_question']
    sq = row['SA_question']

    uq_prompt = f"""You are a question answering model. Your task is to determine whether the following question is answerable using only the information provided in the context.

    Instructions:
    - Only output the answers in the format shown below.
    - Do NOT repeat the questions.
    - If the context contains the answer, extract it exactly as it appears in the context (copy the exact span). Do not use any external knowledge.
    - If the context does not contain the answer, respond with exactly: Null

    ---

    Context:
    {context}

    Question:
    {uq}

    Respond strictly in this format:

    Answer: <your answer here>
    """

    sq_prompt = f"""You are a question answering model. Your task is to determine whether the following question is answerable using only the information provided in the context.

    Instructions:
    - Only output the answers in the format shown below.
    - Do NOT repeat the questions.
    - If the context contains the answer, extract it exactly as it appears in the context (copy the exact span). Do not use any external knowledge.
    - If the context does not contain the answer, respond with exactly: Null

    ---

    Context:
    {context}

    Question:
    {sq}

    Respond strictly in this format:

    Answer: <your answer here>
    """


    try:
        llama_answer_uq = generate_vote(uq_prompt)
        llama_answer_sq = generate_vote(sq_prompt)
        # print(f"Response: {content}")

    except Exception as e:
        print(f"Error on row {i}: {e}")
        llama_answer = e


    # Append row to output file
    with open(output_file, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([qid, context, label,uq, sq, llama_answer_uq, llama_answer_sq])

    # time.sleep(1)